# N0 — Ingesta y preprocesamiento del corpus (Visualización)
## AI-MELT: Nivel 0 — Exploración visual

**Prerequisito:** Haber ejecutado `N0_corpus_ingestion.ipynb` (procesamiento).

---

### Archivos de entrada esperados (en `./outputs/N0/`)

| Archivo | Columnas clave | Descripción |
|---|---|---|
| `n0_corpus.csv` | ID_corpus, ID_documento, volumen, capitulo, pagina, ID_oracion, oracion_texto, n_palabras, n_caracteres, tokens, lemas, pos_tags, entidades_NER | Una fila por oración con anotaciones lingüísticas |
| `n0_metadata.json` | nombre_corpus, stats, archivos_procesados | Metadatos del corpus |
| `n0_resumen_documentos.csv` | ID_documento, volumen, capitulo, n_oraciones, n_palabras | Estadísticas por documento |

---

### Visualizaciones generadas
1. Estadísticas: oraciones y palabras por volumen y por capítulo
2. Distribución de longitud de oraciones (corpus + por volumen)
3. Entidades nombradas: corpus completo y por volumen
4. Distribución POS: corpus completo y por volumen
5. Wordcloud: corpus completo y por volumen
6. Boxplot de longitud por volumen

## 1. Configuración y carga de datos

In [ ]:
# ============================================================
# 1. IMPORTS Y CARGA
# ============================================================
import gc
import json
import os
import re
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

INPUT_DIR = "./outputs/N0/"
VIZ_DIR = "./outputs/N0/viz/"
os.makedirs(VIZ_DIR, exist_ok=True)

print("Cargando n0_corpus.csv...")
df_n0 = pd.read_csv(os.path.join(INPUT_DIR, "n0_corpus.csv"))
print("✓ Cargado")

print("✓ Cargado: n0_corpus.csv")
print(f"  Oraciones: {len(df_n0):,}")
print(f"  Volúmenes: {df_n0['volumen'].nunique() if 'volumen' in df_n0.columns else 'N/A'}")
print(f"  Capítulos: {df_n0['capitulo'].nunique() if 'capitulo' in df_n0.columns else 'N/A'}")
print(f"  Memoria: {df_n0.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

volumenes = sorted(df_n0["volumen"].unique()) if "volumen" in df_n0.columns else ["corpus"]
capitulos = sorted(df_n0["capitulo"].unique()) if "capitulo" in df_n0.columns else []
n_volumenes = len(volumenes)

vol_colors = sns.color_palette("husl", max(n_volumenes, 1))
vol_color_map = {v: vol_colors[i] for i, v in enumerate(volumenes)}

def safe_json_loads(s):
    try:
        return json.loads(s) if isinstance(s, str) else []
    except (json.JSONDecodeError, TypeError):
        return []

def extract_entities_for_subset(df_subset):
    ents = []
    for ner_json in df_subset["entidades_NER"]:
        for e in safe_json_loads(ner_json):
            ents.append((e.get("text", ""), e.get("label", "")))
    return ents

def extract_pos_for_subset(df_subset):
    pos_list = []
    for pos_json in df_subset["pos_tags"]:
        pos_list.extend(safe_json_loads(pos_json))
    return pos_list

def extract_content_lemmas_for_subset(df_subset):
    lemmas = []
    for lemas_json, pos_json in zip(df_subset["lemas"], df_subset["pos_tags"], strict=False):
        for lemma, pos in zip(safe_json_loads(lemas_json), safe_json_loads(pos_json), strict=False):
            if pos in {"NOUN", "VERB", "ADJ"} and len(lemma) > 2:
                lemmas.append(lemma.lower())
    return lemmas

print(f"✓ Funciones cargadas | Volúmenes: {n_volumenes} | Capítulos: {len(capitulos)}")

## 8. Visualizaciones exploratorias

Visualizaciones a dos niveles: **corpus completo** y **por volumen**. Para las vistas por volumen se genera un subplot por cada volumen del Informe.

> **Nota sobre memoria:** Las funciones de visualización extraen datos bajo demanda desde el DataFrame y liberan memoria tras cada gráfico.

In [ ]:
# ============================================================
# 8.0 FUNCIONES AUXILIARES PARA VISUALIZACIÓN
# ============================================================

volumenes = sorted(df_n0["volumen"].unique())
capitulos = sorted(df_n0["capitulo"].unique())
n_volumenes = len(volumenes)

# Paleta de colores por volumen
vol_colors = sns.color_palette("husl", n_volumenes)
vol_color_map = {v: vol_colors[i] for i, v in enumerate(volumenes)}

def safe_json_loads(s):
    """Parse JSON string de forma segura."""
    try:
        return json.loads(s) if isinstance(s, str) else []
    except (json.JSONDecodeError, TypeError):
        return []

def extract_entities_for_subset(df_subset):
    """Extrae entidades NER de un subconjunto del DataFrame sin cargar todo en memoria."""
    ents = []
    for ner_json in df_subset["entidades_NER"]:
        for e in safe_json_loads(ner_json):
            ents.append((e.get("text", ""), e.get("label", "")))
    return ents

def extract_pos_for_subset(df_subset):
    """Extrae POS tags de un subconjunto."""
    pos_list = []
    for pos_json in df_subset["pos_tags"]:
        pos_list.extend(safe_json_loads(pos_json))
    return pos_list

def extract_content_lemmas_for_subset(df_subset):
    """Extrae lemas de contenido (NOUN, VERB, ADJ) para wordcloud."""
    lemmas = []
    for lemas_json, pos_json in zip(df_subset["lemas"], df_subset["pos_tags"], strict=False):
        lemas_list = safe_json_loads(lemas_json)
        pos_list = safe_json_loads(pos_json)
        for lemma, pos in zip(lemas_list, pos_list, strict=False):
            if pos in {"NOUN", "VERB", "ADJ"} and len(lemma) > 2:
                lemmas.append(lemma.lower())
    return lemmas

print("✓ Funciones de visualización cargadas")
print(f"  Volúmenes: {n_volumenes} — {volumenes}")
print(f"  Capítulos: {len(capitulos)}")

### 8.1 Estadísticas: oraciones y palabras por capítulo y por volumen

In [ ]:
# ============================================================
# 8.1 ORACIONES Y PALABRAS — POR VOLUMEN Y POR CAPÍTULO
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  CONFIGURACIÓN                                          │
# └─────────────────────────────────────────────────────────┘

# Para los gráficos por volumen individual:
# "magnitud"  → barras ordenadas de mayor a menor
# "aparicion" → barras en el orden en que aparecen los capítulos en el volumen
ORDEN_CAPITULOS = "aparicion"

# ── Preparar datos ──

vol_stats = df_n0.groupby("volumen").agg(
    n_oraciones=("ID_oracion", "count"),
    n_palabras=("n_palabras", "sum")
).reindex(volumenes)

cap_stats = df_n0.groupby(["volumen", "capitulo"]).agg(
    n_oraciones=("ID_oracion", "count"),
    n_palabras=("n_palabras", "sum"),
    primera_pagina=("pagina", "min")
).reset_index()

display_caps = cap_stats.sort_values("n_oraciones", ascending=True).tail(30)

# ── Gráfico 1: Oraciones por volumen ──
fig, ax = plt.subplots(figsize=(12, max(4, n_volumenes * 0.8)))
ax.barh([v[:40] for v in volumenes], vol_stats["n_oraciones"].values,
        color=[vol_color_map[v] for v in volumenes])
ax.set_xlabel("Oraciones")
ax.set_title("Oraciones por volumen")
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_oraciones_por_volumen.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# ── Gráfico 2: Palabras por volumen ──
fig, ax = plt.subplots(figsize=(12, max(4, n_volumenes * 0.8)))
ax.barh([v[:40] for v in volumenes], vol_stats["n_palabras"].values,
        color=[vol_color_map[v] for v in volumenes])
ax.set_xlabel("Palabras")
ax.set_title("Palabras por volumen")
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_palabras_por_volumen.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# ── Gráfico 3: Oraciones por capítulo (todos los volúmenes, con color por volumen) ──
fig, ax = plt.subplots(figsize=(14, max(8, len(display_caps) * 0.45)))
labels = [f"{row['capitulo'][:40]}  [{row['volumen'][:20]}]" for _, row in display_caps.iterrows()]
colors = [vol_color_map.get(row["volumen"], "#888") for _, row in display_caps.iterrows()]
ax.barh(labels, display_caps["n_oraciones"].values, color=colors)
ax.set_xlabel("Oraciones")
ax.set_title(f"Oraciones por capítulo — todos los volúmenes (top {len(display_caps)})")
ax.tick_params(axis='y', labelsize=7)
# Leyenda de volúmenes
from matplotlib.patches import Patch

vol_in_display = display_caps["volumen"].unique()
legend_patches = [Patch(facecolor=vol_color_map.get(v, "#888"), label=v[:30]) for v in vol_in_display]
ax.legend(handles=legend_patches, loc='lower right', fontsize=7, title="Volumen")
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_oraciones_por_capitulo_global.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# ── Gráfico 4: Palabras por capítulo (todos los volúmenes, con color por volumen) ──
display_caps_pal = cap_stats.sort_values("n_palabras", ascending=True).tail(30)
fig, ax = plt.subplots(figsize=(14, max(8, len(display_caps_pal) * 0.45)))
labels = [f"{row['capitulo'][:40]}  [{row['volumen'][:20]}]" for _, row in display_caps_pal.iterrows()]
colors = [vol_color_map.get(row["volumen"], "#888") for _, row in display_caps_pal.iterrows()]
ax.barh(labels, display_caps_pal["n_palabras"].values, color=colors)
ax.set_xlabel("Palabras")
ax.set_title(f"Palabras por capítulo — todos los volúmenes (top {len(display_caps_pal)})")
ax.tick_params(axis='y', labelsize=7)
vol_in_display2 = display_caps_pal["volumen"].unique()
legend_patches2 = [Patch(facecolor=vol_color_map.get(v, "#888"), label=v[:30]) for v in vol_in_display2]
ax.legend(handles=legend_patches2, loc='lower right', fontsize=7, title="Volumen")
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_palabras_por_capitulo_global.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# ── Gráficos 5+: Por cada volumen, oraciones y palabras por capítulo ──
for vol in volumenes:
    vol_data = cap_stats[cap_stats["volumen"] == vol].copy()
    if len(vol_data) == 0:
        continue

    if ORDEN_CAPITULOS == "magnitud":
        vol_data_orac = vol_data.sort_values("n_oraciones", ascending=True)
        vol_data_pal = vol_data.sort_values("n_palabras", ascending=True)
    else:
        vol_data_orac = vol_data.sort_values("primera_pagina", ascending=False)
        vol_data_pal = vol_data.sort_values("primera_pagina", ascending=False)

    color = vol_color_map.get(vol, "#888")
    n_caps = len(vol_data)

    # Oraciones
    fig, ax = plt.subplots(figsize=(12, max(4, n_caps * 0.5)))
    ax.barh([c[:50] for c in vol_data_orac["capitulo"]], vol_data_orac["n_oraciones"].values, color=color)
    ax.set_xlabel("Oraciones")
    ax.set_title(f"Oraciones por capítulo — {vol[:50]} (orden: {ORDEN_CAPITULOS})")
    ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()
    vol_slug = re.sub(r'[^a-zA-Z0-9]', '_', vol[:30]).lower()
    plt.savefig(os.path.join(VIZ_DIR, f"viz_oraciones_{vol_slug}.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    # Palabras
    fig, ax = plt.subplots(figsize=(12, max(4, n_caps * 0.5)))
    ax.barh([c[:50] for c in vol_data_pal["capitulo"]], vol_data_pal["n_palabras"].values, color=color)
    ax.set_xlabel("Palabras")
    ax.set_title(f"Palabras por capítulo — {vol[:50]} (orden: {ORDEN_CAPITULOS})")
    ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"viz_palabras_{vol_slug}.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

gc.collect()
print(f"✓ {2 + 2 + n_volumenes * 2} gráficos generados (orden capítulos: {ORDEN_CAPITULOS})")

### 8.2 Distribución de longitud de oraciones

In [ ]:
# ============================================================
# 8.2 DISTRIBUCIÓN DE LONGITUD — CORPUS + POR VOLUMEN
# ============================================================

# --- Corpus completo ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df_n0["n_palabras"], bins=50, color="#3B8BD4", edgecolor="white", alpha=0.8)
med = df_n0["n_palabras"].median()
ax.axvline(med, color="red", linestyle="--", label=f"Mediana: {med:.0f}")
ax.set_title("Distribución de longitud de oraciones — CORPUS COMPLETO")
ax.set_xlabel("Palabras por oración")
ax.set_ylabel("Frecuencia")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_longitud_corpus.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# --- Por volumen ---
for vol in volumenes:
    data = df_n0[df_n0["volumen"] == vol]["n_palabras"]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(data, bins=40, color=vol_color_map[vol], edgecolor="white", alpha=0.8)
    med_v = data.median()
    ax.axvline(med_v, color="red", linestyle="--", label=f"Mediana: {med_v:.0f}")
    ax.set_title(f"Distribución de longitud de oraciones — {vol[:50]}")
    ax.set_xlabel("Palabras por oración")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)
    plt.tight_layout()
    vol_slug = re.sub(r'[^a-zA-Z0-9]', '_', vol[:30]).lower()
    plt.savefig(os.path.join(VIZ_DIR, f"viz_longitud_{vol_slug}.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

gc.collect()

### 8.3 Entidades nombradas: corpus completo y por volumen

In [ ]:
# ============================================================
# 8.3 ENTIDADES NOMBRADAS — CORPUS COMPLETO + POR VOLUMEN
# ============================================================

def plot_ner_for_subset(df_subset, label, color="#D85A30", vol_slug="corpus"):
    """Extrae NER de un subconjunto y genera dos gráficos independientes."""
    ents = extract_entities_for_subset(df_subset)
    if not ents:
        print(f"  ⚠ Sin entidades NER para {label}")
        return
    df_ents = pd.DataFrame(ents, columns=["text", "label"])
    print(f"  {label}: {len(df_ents):,} entidades | {df_ents['text'].nunique():,} únicas")

    # Top 20 entidades
    fig, ax = plt.subplots(figsize=(12, 8))
    top = df_ents["text"].value_counts().head(20)
    ax.barh(top.index[::-1], top.values[::-1], color=color)
    ax.set_xlabel("Frecuencia")
    ax.set_title(f"Top 20 entidades nombradas — {label}")
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"viz_NER_top20_{vol_slug}.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    # Distribución por tipo
    fig, ax = plt.subplots(figsize=(8, 8))
    tc = df_ents["label"].value_counts()
    ax.pie(tc.values, labels=tc.index, autopct='%1.1f%%',
           colors=sns.color_palette("husl", len(tc)), startangle=90)
    ax.set_title(f"Distribución por tipo NER — {label}")
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"viz_NER_tipos_{vol_slug}.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    del df_ents, ents

# Corpus completo
print("Extrayendo entidades NER...")
plot_ner_for_subset(df_n0, "CORPUS COMPLETO", color="#D85A30", vol_slug="corpus")

# Por volumen
for vol in volumenes:
    vol_slug = re.sub(r'[^a-zA-Z0-9]', '_', vol[:30]).lower()
    color = vol_color_map.get(vol, "#888")
    plot_ner_for_subset(df_n0[df_n0["volumen"] == vol], vol[:50], color=color, vol_slug=vol_slug)

gc.collect()

### 8.4 Distribución de categorías gramaticales (POS): corpus y por volumen

In [ ]:
# ============================================================
# 8.4 POS — CORPUS COMPLETO + POR VOLUMEN
# ============================================================

def plot_pos_for_subset(df_subset, label, color="#3B8BD4", vol_slug="corpus"):
    pos_counts = Counter(extract_pos_for_subset(df_subset))
    pos_top = pd.DataFrame(pos_counts.most_common(12), columns=["POS", "Freq"])
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(pos_top["POS"], pos_top["Freq"], color=color, edgecolor="white")
    ax.set_title(f"Distribución POS — {label}")
    ax.set_xlabel("Categoría gramatical")
    ax.set_ylabel("Frecuencia")
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"viz_pos_{vol_slug}.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    del pos_counts

# Corpus completo
print("Extrayendo POS...")
plot_pos_for_subset(df_n0, "CORPUS COMPLETO", color="#3B8BD4", vol_slug="corpus")

# Por volumen
for vol in volumenes:
    vol_slug = re.sub(r'[^a-zA-Z0-9]', '_', vol[:30]).lower()
    plot_pos_for_subset(df_n0[df_n0["volumen"] == vol], vol[:50], color=vol_color_map.get(vol, "#888"), vol_slug=vol_slug)

# Porcentajes de categorías clave (corpus)
pos_all = Counter(extract_pos_for_subset(df_n0))
total_pos = sum(pos_all.values())
print("\nCategorías clave para metáfora (corpus):")
for pos in ["VERB", "NOUN", "ADJ", "ADV"]:
    pct = pos_all.get(pos, 0) / total_pos * 100
    print(f"  {pos}: {pos_all.get(pos, 0):,} ({pct:.1f}%)")
del pos_all
gc.collect()

### 8.5 Wordcloud: corpus completo y por volumen

In [ ]:
# ============================================================
# 8.5 WORDCLOUD — CORPUS COMPLETO
# ============================================================

lemmas_corpus = extract_content_lemmas_for_subset(df_n0)
if lemmas_corpus:
    wc = WordCloud(width=1200, height=500, background_color="white",
                    max_words=150, colormap="viridis", collocations=False).generate(" ".join(lemmas_corpus))
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Wordcloud — corpus completo (sustantivos, verbos, adjetivos)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_wordcloud_corpus.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    del wc
del lemmas_corpus
gc.collect()

In [ ]:
# ============================================================
# 8.5 WORDCLOUD — CORPUS COMPLETO + POR VOLUMEN
# ============================================================

# --- Corpus completo ---
print("Generando wordclouds...")
lemmas_corpus = extract_content_lemmas_for_subset(df_n0)
if lemmas_corpus:
    wc = WordCloud(width=1200, height=500, background_color="white",
                    max_words=150, colormap="viridis", collocations=False).generate(" ".join(lemmas_corpus))
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Wordcloud — CORPUS COMPLETO (sustantivos, verbos, adjetivos)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_wordcloud_corpus.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    del wc
del lemmas_corpus

# --- Por volumen ---
for vol in volumenes:
    lemmas_vol = extract_content_lemmas_for_subset(df_n0[df_n0["volumen"] == vol])
    if lemmas_vol and len(lemmas_vol) > 10:
        wc = WordCloud(width=800, height=400, background_color="white",
                        max_words=80, colormap="viridis", collocations=False).generate(" ".join(lemmas_vol))
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Wordcloud — {vol[:50]}", fontsize=12)
        plt.tight_layout()
        vol_slug = re.sub(r'[^a-zA-Z0-9]', '_', vol[:30]).lower()
        plt.savefig(os.path.join(VIZ_DIR, f"viz_wordcloud_{vol_slug}.png"), dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        del wc
    else:
        print(f"⚠ Sin lemas suficientes para wordcloud de {vol[:40]}")
    del lemmas_vol

gc.collect()

### 8.6 Boxplot de longitud de oraciones por volumen

In [ ]:
# ============================================================
# 8.6 BOXPLOT DE LONGITUD POR VOLUMEN
# ============================================================

if n_volumenes > 1:
    fig, ax = plt.subplots(figsize=(12, 6))
    data_by_vol = [df_n0[df_n0["volumen"] == v]["n_palabras"].values for v in volumenes]
    bp = ax.boxplot(data_by_vol, labels=[v[:25] for v in volumenes], patch_artist=True)
    for patch, color in zip(bp['boxes'], vol_colors, strict=False):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xlabel("Volumen")
    ax.set_ylabel("Palabras por oración")
    ax.set_title("Distribución de longitud de oraciones por volumen")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_longitud_por_volumen.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
gc.collect()

## 9. Resumen

In [ ]:
# ============================================================
# 9. RESUMEN
# ============================================================
print("=" * 60)
print("N0 — VISUALIZACIONES COMPLETADAS")
print("=" * 60)
print(f"\nArchivos en {VIZ_DIR}:")
for f_name in sorted(os.listdir(VIZ_DIR)):
    size = os.path.getsize(os.path.join(VIZ_DIR, f_name)) / 1024
    icon = "🖼"
    print(f"  {icon} {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: Ejecutar N1_metafora_primaria.ipynb")